# B05：分类模型比较与错误分析（辅修学生练习）

T10 后 2 实验学时。四个预设模型、相同三折、相同主指标，不追加搜索。Boosting 运行代码已提供，不要求实现训练器。

原 120 行 train 作开发内交叉验证，旧 40 行 validation 不评价。最后 20 分钟将项目基线片段追加到原项目 Notebook，不覆盖草稿。

## 1. 数据与比较条件（已提供）

只使用开工前可用的两特征；填补在折内 fit，逻辑回归额外使用折内标准化。所有候选使用相同三折，数值树无需强行加标准化。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import confusion_matrix

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend(
        [
            parent / "data/boosting-classification.csv",
            parent / "units/boosting/data/boosting-classification.csv",
        ]
    )
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("请保留目录关系，从学生包根目录或labs启动")
raw = pd.read_csv(data_path)
dev = raw[raw.split == "train"].reset_index(drop=True)
features = ["planned_load_ratio", "ambient_index"]
X = dev[features]
y = dev.over_limit.to_numpy()
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y))
assert X.shape == (120, 2)
display(dev.head())

## 2. 四个模型（全部提供）

Logistic C=1；Tree 深度 5；Forest 50 棵、每树深度 5；Boosting 50 轮、每个弱树深度 1、学习率 0.1。浅树与深树并非等参数容量，这里比较固定教学方案，不宣称最优调参排名。

In [ ]:
def build_models():
    return {
        "Logistic": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(C=1.0, max_iter=1000)),
            ]
        ),
        "Tree": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "model",
                    DecisionTreeClassifier(
                        max_depth=5, min_samples_leaf=2, random_state=42
                    ),
                ),
            ]
        ),
        "Forest": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=50,
                        max_depth=5,
                        min_samples_leaf=2,
                        max_features=1,
                        bootstrap=True,
                        random_state=42,
                        n_jobs=1,
                    ),
                ),
            ]
        ),
        "Boosting": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "model",
                    GradientBoostingClassifier(
                        loss="log_loss",
                        n_estimators=50,
                        learning_rate=0.1,
                        max_depth=1,
                        min_samples_leaf=2,
                        random_state=42,
                    ),
                ),
            ]
        ),
    }


models = build_models()

比较四个已给方案，先预测复杂模型是否一定更好，再看结果。

## 3. 统一验证

完成 evaluate(model)，调用 cross_validate，传完整 Pipeline、原始 X/y、共同 folds，scoring="accuracy"、return_train_score=True、return_estimator=True、error_score="raise"。test_score 在这里是折内验证，不是最终测试。

In [ ]:
def evaluate(model):
    # TODO：补齐验证调用，完成后删除停止提示。
    raise NotImplementedError("请完成同折评价")
    return cross_validate(
        ...,
        ...,
        ...,
        cv=...,
        scoring="accuracy",
        return_train_score=True,
        return_estimator=True,
        error_score="raise",
    )

定义 error_counts(actual,pred)，返回 TP/FP/FN/TN 字典；confusion_matrix(...,labels=[0,1]).ravel()按 TN、FP、FN、TP 排列，注意顺序。

In [ ]:
def error_counts(actual, pred):
    # TODO：补齐真实值和预测，完成后删除停止提示。
    raise NotImplementedError("请完成混淆计数")
    tn, fp, fn, tp = confusion_matrix(..., ..., labels=[0, 1]).ravel()
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}

In [ ]:
assert error_counts([1, 1, 0, 0], [1, 0, 1, 0]) == {"TP": 1, "FP": 1, "FN": 1, "TN": 1}
runs = {}
oof = {}
rows = []
for name, model in models.items():
    run = evaluate(model)
    runs[name] = run
    pred = np.full(len(y), -1, dtype=int)
    coverage = np.zeros(len(y), dtype=int)
    for (ti, vi), fitted in zip(folds, run["estimator"]):
        np.testing.assert_allclose(
            fitted.named_steps["imputer"].statistics_, X.iloc[ti].median().to_numpy()
        )
        pred[vi] = fitted.predict(X.iloc[vi])
        coverage[vi] += 1
    np.testing.assert_array_equal(coverage, np.ones(len(y)))
    oof[name] = pred
    scores = run["test_score"]
    rows.append(
        {
            "model": name,
            "train_mean": run["train_score"].mean(),
            "fold1": scores[0],
            "fold2": scores[1],
            "fold3": scores[2],
            "cv_mean": scores.mean(),
            "score_std": scores.std(ddof=0),
            **error_counts(y, pred),
        }
    )
results = pd.DataFrame(rows)
display(results)
for name in models:
    np.testing.assert_allclose(
        results.set_index("model").loc[name, "cv_mean"], np.mean(oof[name] == y)
    )
fig, ax = plt.subplots(figsize=(7, 3.3))
results.set_index("model")[["train_mean", "cv_mean"]].plot.bar(ax=ax, rot=0)
ax.set(ylabel="Accuracy", ylim=(0, 1))
fig.tight_layout()
plt.show()

## 4. 错误复核与解释（抽取代码提供）

每个模型按编号展示前两条错误，不挑有利案例；没有错误时显示空表。score_std 只描述三折波动，不是置信区间。

In [ ]:
for name in models:
    examples = dev[["batch_id", *features, "over_limit"]].copy()
    examples["prediction"] = oof[name]
    print(name)
    display(
        examples[examples.prediction != examples.over_limit]
        .sort_values("batch_id")
        .head(2)
    )

回答：

1. 引用 CV 均分、FP/FN 说明比较结果；是否能证明某方法普遍更好？
2. 选一个实际错误，区分误报/漏报，说明可能代价；不能凭一行断言原因。
3. Boosting 后续弱树修正已有模型，与森林的组合方式有什么不同？

**我的回答：**（填写）

## 5. 综合任务基线（20 分钟）

打开 project/README.md，将 project/baseline-cells.ipynb 的单元格追加到本人原项目 Notebook。先确认原计划的划分模式、种子和主指标，再运行一个多数类基线；不得读保留测试、不新建替代项目、不改挑有利指标。

保存实验 Notebook 与原项目 Notebook，均放非保护盘并另留副本。